In [2]:
import chromadb

DATA_DIR = "../data"

# Conectando ao Docker do ChromaDB
print("Conectando ao ChromaDB no Docker (localhost:8001)...")
cliente_chroma = chromadb.HttpClient(host="localhost", port=8001)
print("✅ Conexão estabelecida!")

Conectando ao ChromaDB no Docker (localhost:8001)...
✅ Conexão estabelecida!


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import json
from langchain_core.documents import Document

print("A carregar os ficheiros da pasta data...\n")

loader = DirectoryLoader(
    DATA_DIR, 
    glob="**/*.*",
    exclude=["dataset_finetuning_postech.jsonl", "dataset_rag.json"], 
    loader_cls=TextLoader
)

raw_documents = loader.load()

file_names = [doc.metadata['source'] for doc in raw_documents]
print(f"✅ {len(raw_documents)} ficheiro(s) carregado(s):")
for name in file_names:
    print(f" - {name}")
print("\n")

headers_to_split_on = [
        ("#", "Título Principal"),
        ("##", "Secção"),
        ("###", "Sub-secção"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

md_chunks = []
for doc in raw_documents:
    chunks = markdown_splitter.split_text(doc.page_content)
    for chunk in chunks:
        chunk.metadata.update(doc.metadata) 
    md_chunks.extend(chunks)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
final_chunks = text_splitter.split_documents(md_chunks)

print(f"✅ O texto total foi dividido em {len(final_chunks)} fragmentos otimizados.")

/Users/pedromarquardt/dev/estudos/tech-challenger/medical-assistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


A carregar os ficheiros da pasta data...

✅ 5 ficheiro(s) carregado(s):
 - ../data/Diretriz_Dor_Toracica_IAM.md
 - ../data/Protocolo_Cetoacidose_Diabetica.md
 - ../data/Protocolo_Choque_Anafilatico.md
 - ../data/Protocolo_Crise_Asma.md
 - ../data/Diretriz_Profilaxia_Tetano.md


✅ O texto total foi dividido em 35 fragmentos otimizados.


In [5]:
import chromadb
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

print("A conectar ao ChromaDB no Docker (localhost:8001)...")
chroma_client = chromadb.HttpClient(host="localhost", port=8001)
print("✅ Conexão estabelecida!\n")

print("A gerar embeddings com o modelo bge-m3 e a enviar para o Docker...\n")

embeddings_model = OllamaEmbeddings(model="bge-m3")
collection_name = "hospital_protocols"

try:
    chroma_client.delete_collection(collection_name)
    print(f"Coleção '{collection_name}' anterior apagada para evitar duplicatas.")
except Exception:
    pass 

vectorstore = Chroma.from_documents(
    documents=final_chunks, 
    embedding=embeddings_model,
    client=chroma_client,
    collection_name=collection_name
)

print(f"🚀 SUCESSO! Dados inseridos na coleção '{collection_name}' com sucesso.")

A conectar ao ChromaDB no Docker (localhost:8001)...
✅ Conexão estabelecida!

A gerar embeddings com o modelo bge-m3 e a enviar para o Docker...

Coleção 'hospital_protocols' anterior apagada para evitar duplicatas.


/var/folders/bk/c5_22m9n505fd69n9tdv0cb80000gp/T/ipykernel_82978/2578499665.py:11: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings_model = OllamaEmbeddings(model="bge-m3")


🚀 SUCESSO! Dados inseridos na coleção 'hospital_protocols' com sucesso.


In [6]:
# Verificar coleções existentes
print("🔍 Verificando coleções no ChromaDB:")
collections = chroma_client.list_collections()
for collection in collections:
    print(f"  - Nome: {collection.name}")
    print(f"    ID: {collection.id}")
    print(f"    Count: {collection.count()}")

🔍 Verificando coleções no ChromaDB:
  - Nome: hospital_protocols
    ID: dab1ccaa-c95e-4d63-9e2b-6b1e64c87f29
    Count: 35


In [7]:
# Testar busca na coleção
print("🔍 Testando busca na coleção hospital_protocols:")
vectorstore.similarity_search("diabetes", k=3)

🔍 Testando busca na coleção hospital_protocols:


[Document(id='4d52eb6b-1d4d-4beb-8e03-a5aef5b5f8ed', metadata={'Sub-secção': '1.0 Objetivo', 'Título Principal': 'Protocolo Interno para Manejo de Cetoacidose Diabética (CAD) em Adultos', 'source': '../data/Protocolo_Cetoacidose_Diabetica.md'}, page_content='Padronizar o diagnóstico e o manejo de pacientes adultos com Cetoacidose Diabética (CAD) admitidos em nossas unidades, visando a rápida estabilização metabólica, a prevenção de complicações e a otimização do tempo de internação em Unidade de Terapia Intensiva (UTI).'),
 Document(id='1c97e1b7-7a59-4881-b0fa-0a55f8765994', metadata={'Título Principal': 'Protocolo Interno para Manejo de Cetoacidose Diabética (CAD) em Adultos', 'source': '../data/Protocolo_Cetoacidose_Diabetica.md', 'Sub-secção': '2.0 Âmbito de Aplicação'}, page_content='Este protocolo aplica-se a todos os pacientes adultos (idade ≥ 18 anos) diagnosticados com Cetoacidose Diabética e atendidos no Pronto-Socorro, Enfermarias e UTI deste hospital.'),
 Document(id='c9b561